In [ ]:
# Imports
import time
from astroquery.gaia import Gaia

# 1. Log in to the Gaia Archive
# You can also use Gaia.login() for a popup
Gaia.login(user='YourAccount', password='YourPassword')

# Columns
columns = """
    source_id, ra, dec, parallax, parallax_error, parallax_over_error, 
    pmra, pmra_error, pmdec, pmdec_error, phot_g_mean_mag, 
    phot_bp_mean_flux, phot_bp_mean_flux_error, phot_rp_mean_flux, 
    phot_rp_mean_flux_error, bp_rp, radial_velocity, non_single_star, 
    has_epoch_rv, teff_gspphot, logg_gspphot, distance_gspphot, ebpminrp_gspphot
"""

# Query with b > 30 (above the galactic plane) and 30% parallax uncertainty filter
# TOP 100000000 to bypass the default row limit
query = f"""
SELECT TOP 100000000
    {columns}
FROM gaiadr3.gaia_source
WHERE
    b > 30
    AND parallax_over_error >= 3.33
    AND parallax > 0
"""

# Set the row limit to unlimited for the current account session
Gaia.ROW_LIMIT = -1
print("Starting asynchronous query for b > 30 with 30% filter...")

# 2. Launch the job in the background due to server instability
job = Gaia.launch_job_async(query, background=True)
print(f"Job ID: {job.jobid}")

# 3. Wait until the job is complete
while job.get_phase() not in ['COMPLETED', 'ERROR', 'ABORTED']:
    print(f"Status: {job.get_phase()}, waiting...")
    time.sleep(60)
print(f"Final status: {job.get_phase()}")

# 4. Retrieve results and save to file
if job.get_phase() == 'COMPLETED':
    result_table = job.get_results()
    output_name = "gaia_b_boven30deg_30per_DR3.csv"
    result_table.write(output_name, format='ascii.csv', overwrite=True)
    print(f"--- DONE ---")
    print(f"Total number of stars found: {len(result_table)}")
    print(f"File saved as: {output_name}")
else:
    print(f"Error: {job.get_error()}")

# 5. Log out
Gaia.logout()